# Lesson 08 - Multi-Agent Design Pattern

## Setup

In [1]:
import logging
import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
import sys
from pathlib import Path

repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared").exists()), Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from shared.agent_provider import create_provider, describe_provider


<frozen abc>:106: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.
<frozen abc>:106: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.


In [ ]:
provider = create_provider()
print(f"Provider configured: {describe_provider(provider)}")

Provider configured: OpenAI-compatible | model=qwen3.6-plus | endpoint=https://dashscope-intl.aliyuncs.com/compatible-mode/v1


## Why Multi-Agent Systems?

Real-world tasks like trip planning involve many different kinds of expertise — logistics, local knowledge, budgeting, and more. A single agent trying to handle everything quickly becomes unwieldy.

Multi-agent systems solve this through **specialization**: each agent focuses on one area of expertise, producing higher-quality results than a generalist. They also improve **scalability** — you can add new agents (e.g., a flight specialist, a restaurant critic) without rewriting the existing workflow. The agents compose together through a structured pipeline, passing context from one to the next.

## Creating Specialized Agents

In [3]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Building a Sequential Workflow

`WorkflowBuilder` lets you wire agents into a directed graph. Here we create a simple two-step pipeline: the **TravelPlanner** drafts the itinerary, then the **TravelConcierge** reviews and enhances it.

In [4]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

/var/folders/s3/4tvg2_0x6kgddy1k1ck_nxr40000gp/T/ipykernel_73485/1028293691.py:3: DeprecationWarning: WorkflowBuilder built without explicit output_from or intermediate_output_from; every yield_output produces type='output' for compatibility. Pass output_from='all', output_from=[...], or intermediate_output_from=[...] to opt into explicit designation - explicit designation will be required in a future version.
  .build()



🤖 TravelPlanner:
Here’s a carefully curated **5-day Paris itinerary** designed specifically for a food-loving couple, optimized to stay within a **$3,000 USD total budget** (excluding international flights). The plan balances iconic culinary experiences, neighborhood exploration, hands-on learning, and smart budgeting without sacrificing quality.

---
### 📍 WHERE TO STAY (Budget-Friendly Food Hubs)
**Recommended neighborhoods:** 10th, 11th, or 12th arrondissements. You’ll find excellent value, walkable access to markets, bistros, and the Canal Saint-Martin, plus easy Metro links.
- Target: 3★ boutique hotel, guesthouse, or legal Airbnb: **$110–$140/night**

---
## 🗓️ DAILY ITINERARY

### **DAY 1: Arrival & The Parisian Bakery & Bistro Basics**
- **14:00** | Check in, unpack, refresh
- **15:30** | **Bakery crawl** on Rue Oberkampf / Rue des Martyrs. Start at **Boulangerie Bo** or **Du Pain et des Idées** for a seasonal fruit tart, classic croissant, and a *café allongé*. (~€10/person)


## Adding More Agents to the Workflow

One of the biggest advantages of the multi-agent pattern is how easy it is to extend. Below we add a **BudgetReviewer** agent that checks the plan against the traveler's budget, flags items that might push costs over the limit, and suggests money-saving alternatives. The workflow now runs three agents in sequence:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```

In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Summary

In this lesson you learned how to:

1. **Create specialized agents** — each with a focused role (planning, concierge, budget review).
2. **Wire agents into a sequential workflow** using `WorkflowBuilder` and `add_edge`.
3. **Stream output** from a multi-agent pipeline, tracking which agent is speaking.
4. **Extend a workflow** by adding new agents to the chain without modifying existing ones.

The multi-agent design pattern keeps each agent simple while producing richer, more thoroughly reviewed results than any single agent could achieve alone.